# 案例四：陪恩恩老師測試連線棋策略

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johnnychao/python-machine-learning-2026-student/blob/main/notebooks/ai_solution_practicum/04_rl_strategy_story.ipynb)

遊戲社想先用模擬對局檢查簡單策略，再決定要不要投入更複雜的
reinforcement learning。今天先建立可重複的評估環境。

**情境問題：** 只把隨機選欄改成「先贏、再擋、再靠中間」的一步策略，
對隨機對手的勝率如何改變？

- baseline：隨機合法動作。
- candidate：只改成一步攻防策略。
- 對手、棋盤、場數與亂數種子完全相同。
- [Kaggle 題目出處：Connect X]
  (https://www.kaggle.com/competitions/connectx)

執行時只用 Python 與 NumPy，不需安裝競賽環境。


## 1. 建立純 Python 棋盤


In [ ]:
from pathlib import Path
import json
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROWS = 6
COLUMNS = 7
CONNECT = 4
GAMES = 300
BASE_SEED = 20260719


def new_board() -> np.ndarray:
    return np.zeros((ROWS, COLUMNS), dtype=np.int8)


def valid_columns(board: np.ndarray) -> list[int]:
    return [column for column in range(COLUMNS) if board[0, column] == 0]


def drop_piece(board: np.ndarray, column: int, player: int) -> np.ndarray:
    if column not in valid_columns(board):
        raise ValueError(f"欄位 {column} 已滿")
    updated = board.copy()
    for row in range(ROWS - 1, -1, -1):
        if updated[row, column] == 0:
            updated[row, column] = player
            return updated
    raise RuntimeError("找不到可下的位置")


def has_won(board: np.ndarray, player: int) -> bool:
    directions = [(0, 1), (1, 0), (1, 1), (1, -1)]
    for row in range(ROWS):
        for column in range(COLUMNS):
            if board[row, column] != player:
                continue
            for row_step, column_step in directions:
                if all(
                    0 <= row + step * row_step < ROWS
                    and 0 <= column + step * column_step < COLUMNS
                    and board[
                        row + step * row_step,
                        column + step * column_step,
                    ]
                    == player
                    for step in range(CONNECT)
                ):
                    return True
    return False


## 2. 定義兩種策略

candidate 只多看一步：能贏就贏、對手下一步能贏就擋，
否則偏好中間欄位。它不是完整的強化學習模型。


In [ ]:
def choose_column(
    board: np.ndarray,
    player: int,
    rng: random.Random,
    strategy: str,
) -> int:
    valid = valid_columns(board)
    if not valid:
        raise ValueError("棋盤已滿")

    if strategy == "random":
        return rng.choice(valid)

    if strategy != "one_step":
        raise ValueError(f"未知策略：{strategy}")

    opponent = 1 if player == 2 else 2
    for column in valid:
        if has_won(drop_piece(board, column, player), player):
            return column
    for column in valid:
        if has_won(drop_piece(board, column, opponent), opponent):
            return column

    center_distance = {
        column: abs(column - (COLUMNS - 1) / 2)
        for column in valid
    }
    best_distance = min(center_distance.values())
    best_columns = [
        column
        for column, distance in center_distance.items()
        if distance == best_distance
    ]
    return rng.choice(best_columns)


def play_game(
    agent_strategy: str,
    seed: int,
    agent_starts: bool,
) -> tuple[str, np.ndarray]:
    rng = random.Random(seed)
    board = new_board()
    agent_player = 1 if agent_starts else 2
    current_player = 1

    while valid_columns(board):
        strategy = agent_strategy if current_player == agent_player else "random"
        column = choose_column(board, current_player, rng, strategy)
        board = drop_piece(board, column, current_player)

        if has_won(board, current_player):
            result = "win" if current_player == agent_player else "loss"
            return result, board

        current_player = 1 if current_player == 2 else 2

    return "draw", board


## 3. 先做一局健全性檢查


In [ ]:
sample_result, sample_board = play_game(
    agent_strategy="one_step",
    seed=BASE_SEED,
    agent_starts=True,
)
print("範例結果:", sample_result)
print(sample_board)
assert sample_result in {"win", "loss", "draw"}
assert np.isin(sample_board, [0, 1, 2]).all()


## 4. 公平評估 baseline 與 candidate

每一場使用同一組 seed，並交替先後手。唯一改變的是 agent_strategy。


In [ ]:
def evaluate_strategy(strategy: str, games: int = GAMES) -> tuple[dict, np.ndarray]:
    counts = {"win": 0, "loss": 0, "draw": 0}
    first_failure_board = None

    for game_index in range(games):
        result, final_board = play_game(
            agent_strategy=strategy,
            seed=BASE_SEED + game_index,
            agent_starts=(game_index % 2 == 0),
        )
        counts[result] += 1
        if first_failure_board is None and result == "loss":
            first_failure_board = final_board

    rates = {
        "win_rate": counts["win"] / games,
        "loss_rate": counts["loss"] / games,
        "draw_rate": counts["draw"] / games,
        "games": games,
    }
    return rates, first_failure_board


baseline_metrics, baseline_failure = evaluate_strategy("random")
candidate_metrics, candidate_failure = evaluate_strategy("one_step")

comparison = pd.DataFrame(
    [baseline_metrics, candidate_metrics],
    index=["baseline_random", "candidate_one_step"],
)
display(comparison.style.format(
    {
        "win_rate": "{:.1%}",
        "loss_rate": "{:.1%}",
        "draw_rate": "{:.1%}",
        "games": "{:.0f}",
    }
))


## 5. 比較前後並查看失敗棋盤


In [ ]:
win_rate_change = candidate_metrics["win_rate"] - baseline_metrics["win_rate"]
chosen_version = "candidate_one_step" if win_rate_change > 0 else "baseline_random"
print(f"勝率改變：{win_rate_change:+.1%}")
print("依本次模擬勝率暫選：", chosen_version)


def plot_board(board: np.ndarray, title: str) -> None:
    color_map = np.zeros((*board.shape, 3), dtype=float)
    color_map[board == 0] = [0.93, 0.94, 0.95]
    color_map[board == 1] = [0.95, 0.60, 0.07]
    color_map[board == 2] = [0.17, 0.24, 0.31]
    plt.figure(figsize=(7, 5))
    plt.imshow(color_map)
    plt.xticks(range(COLUMNS))
    plt.yticks(range(ROWS))
    plt.grid(color="white", linewidth=2)
    plt.title(title)
    plt.show()


if candidate_failure is not None:
    plot_board(candidate_failure, "candidate 仍然會輸：第一個失敗棋盤")
else:
    print("本次 candidate 沒輸；請增加場數或換對手再測。")


## 6. 限制與人工介入

- 對手只是隨機策略，勝率不能代表面對真人或強代理的表現。
- 一步策略只處理眼前勝負，不會規劃兩步以上的陷阱。
- 模擬場數與先後手都會影響估計，應保留 seed 與場數。
- 若要用在教學排名或正式競賽，仍要由人訂定公平規則與例外處理。

請寫一句結論：目前證據足以進入更複雜模型嗎？下一個只改一項的實驗是什麼？


## 7. 下載實驗紀錄


In [ ]:
experiment_record = {
    "case": "rl_strategy_story",
    "question": "一步攻防策略相較隨機策略，對隨機對手的勝率如何改變？",
    "baseline": {"strategy": "random", "metrics": baseline_metrics},
    "candidate": {"strategy": "one_step", "metrics": candidate_metrics},
    "single_change": "agent 選欄策略",
    "win_rate_change": float(win_rate_change),
    "selected_by_simulation_win_rate": chosen_version,
    "human_review": "公平規則、對手強度、先後手與例外處理",
    "limitation": "只對隨機對手，不代表正式競賽表現",
}

record_path = Path("/content/rl_strategy_experiment.json")
record_path.write_text(
    json.dumps(experiment_record, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(record_path)


In [ ]:
try:
    from google.colab import files
    files.download(str(record_path))
except ImportError:
    print("檔案已保留在", record_path)
